L'objectif de Notebook est d'obtenir 2 modèles d'entrainement  

Le 1er est un Vote Pondéré (Voting / Averaging) de XGBoost et de LightGBM.

Pour avoir au final le résultat final est souvent meilleur que si un seul modèle avait travaillé.


Le 2eme est un modèle de Deep Learning Stochastic gradient descent

In [25]:
import argparse
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.metrics import classification_report, roc_auc_score

Chargement des données

In [ ]:
df = pd.read_csv("/content/maintenance_cleaned.csv")
X = df.drop(columns=['failure_within_24h'])
y = df['failure_within_24h']

Analyse du DataSet

In [ ]:
display(df.head())
display(df.describe())
display(df.info())

,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,failure_within_24h
0,CNC,0.81,49.51,5.10,23.6,860.9,idle,273.80,13.9,0
1,CNC,0.75,40.58,5.30,23.6,899.6,idle,273.85,10.2,0
2,CNC,0.71,49.70,6.43,21.3,862.7,idle,274.15,13.6,0
3,CNC,0.76,43.04,4.79,22.6,870.4,idle,274.55,13.4,0
4,CNC,0.88,41.39,4.44,22.2,881.9,idle,274.70,10.8,0


,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,hours_since_maintenance,ambient_temp,failure_within_24h
count,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000,24042.000000
mean,1.605370,51.357662,8.751044,58.523667,1138.445662,172.630624,12.996398,0.148074
std,1.041905,12.302671,5.300137,38.050389,903.498629,150.722469,2.883994,0.355181
min,0.350000,28.000000,2.200000,10.100000,124.100000,0.000000,8.000000,0.000000
25%,0.850000,42.890000,4.710000,23.100000,496.025000,42.870000,10.500000,0.000000
50%,1.270000,50.060000,6.430000,46.300000,856.000000,121.610000,13.000000,0.000000
75%,2.230000,59.600000,12.990000,90.800000,1667.875000,295.575000,15.500000,0.000000
max,6.370000,95.000000,35.000000,206.500000,4098.800000,575.630000,18.000000,1.000000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24042 entries, 0 to 24041
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   machine_type             24042 non-null  object 
 1   vibration_rms            24042 non-null  float64
 2   temperature_motor        24042 non-null  float64
 3   current_phase_avg        24042 non-null  float64
 4   pressure_level           24042 non-null  float64
 5   rpm                      24042 non-null  float64
 6   operating_mode           24042 non-null  object 
 7   hours_since_maintenance  24042 non-null  float64
 8   ambient_temp             24042 non-null  float64
 9   failure_within_24h       24042 non-null  int64  
dtypes: float64(7), int64(1), object(2)
memory usage: 1.8+ MB


None

On fait un scaling

In [ ]:
# 1. Encodage des colonnes de texte en nombres
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

Séparation des données pour le test et le training.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 97215)

X_train = pd.DataFrame(X_train)

Calcul du poids pour gérer le déséquilibre des classes

In [ ]:
pannes_normales = len(y_train) - sum(y_train)
pannes_reelles = sum(y_train)
ratio_poids = pannes_normales / pannes_reelles

Initialisation des modèles de bases

In [ ]:
# Modele XGBoost
model_xgb = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=ratio_poids
)

# Modèle LightGBM
model_lgb = lgb.LGBMClassifier(
    random_state=42,
    verbose=-1,
    scale_pos_weight=ratio_poids
)

Création et entraînement du Voting Classifier

In [ ]:
modele_ensemble = VotingClassifier(
        estimators=[('xgb', model_xgb), ('lgb', model_lgb)],
        voting='soft'
    )

print("Entraînement du modèle combiné en cours...")
modele_ensemble.fit(X_train, y_train)
print("Entraînement terminé avec succès !")

Entraînement du modèle combiné en cours...
Entraînement terminé avec succès !


Évaluation du modèle

In [ ]:
y_pred = modele_ensemble.predict(X_test)
y_pred_proba = modele_ensemble.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
report = classification_report(y_test, y_pred, output_dict=True)
conf_matrix = confusion_matrix(y_test, y_pred).tolist()

print("\n================ RÉSULTATS DU MODÈLE ================")
print(f"Score ROC AUC : {roc_auc:.4f}")
print(classification_report(y_test, y_pred))


================ RÉSULTATS DU MODÈLE ================
Score ROC AUC : 0.9950
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      5144
           1       0.85      0.95      0.90       867

    accuracy                           0.97      6011
   macro avg       0.92      0.96      0.94      6011
weighted avg       0.97      0.97      0.97      6011



Sauvegarde du modèle avec un fichier.pkl


In [ ]:

nom_fichier_modele = "modele_ensemble_maintenance.pkl"

joblib.dump(modele_ensemble, nom_fichier_modele)
print(f"=> Modèle sauvegardé avec succès sous : {nom_fichier_modele}")

=> Modèle sauvegardé avec succès sous : modele_ensemble_maintenance.pkl


Sauvegarde des métriques au format JSON

In [ ]:
# Sauvegarde des métriques au format JSON
nom_fichier_metrics = "metrics.json"

metrics_summary = {
    "roc_auc": roc_auc,
    "confusion_matrix": conf_matrix,
    "classification_report": report
}

with open(nom_fichier_metrics, 'w') as f:
    json.dump(metrics_summary, f, indent=4)
print(f"=> Métriques exportées sous : {nom_fichier_metrics}")

## Partie 2 : Modèle de Deep Learning (Réseau de neurones avec SGD)

Dans cette section, nous créons un réseau de neurones artificiels (MLP) pour classifier les pannes de maintenance.
Le modèle sera entraîné en utilisant l'optimiseur **SGD (Stochastic Gradient Descent)**. Pour gérer le fort déséquilibre des classes (14,8% de pannes), nous injecterons le poids calculé précédemment (`ratio_poids`) lors de l'entraînement.

In [27]:
model_dl = Sequential([

    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),


    Dense(32, activation='relu'),
    Dropout(0.2),

    # Couche de sortie : 1 neurone avec 'sigmoid' pour sortir une probabilité entre 0 et 1 (binaire)
    Dense(1, activation='sigmoid')
])

# 2. Configuration de l'optimiseur Stochastic Gradient Descent (SGD)
opt = SGD(learning_rate=0.01, momentum=0.9)


model_dl.compile(
    optimizer=opt,
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 4. Préparation des poids des classes pour gérer le déséquilibre
class_weight = {0: 1.0, 1: ratio_poids}

print("Début de l'entraînement du réseau de neurones...")

# 5. Entraînement du modèle
history = model_dl.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weight,
    verbose=1
)

Début de l'entraînement du réseau de neurones...
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


226/226 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4897 - auc: 0.5000 - loss: 691.9110 - val_accuracy: 0.1489 - val_auc: 0.5000 - val_loss: 0.7162
Epoch 2/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4630 - auc: 0.4844 - loss: 1.1807 - val_accuracy: 0.1489 - val_auc: 0.5000 - val_loss: 0.6990
Epoch 3/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4085 - auc: 0.4974 - loss: 1.1801 - val_accuracy: 0.8511 - val_auc: 0.5000 - val_loss: 0.6792
Epoch 4/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6188 - auc: 0.5009 - loss: 1.1799 - val_accuracy: 0.1489 - val_auc: 0.5000 - val_loss: 0.7241
Epoch 5/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4025 - auc: 0.4880 - loss: 1.1808 - val_accuracy: 0.1489 - val_auc: 0.5000 - val_loss: 0.7178
Epoch 6/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4445 - auc: 0.5071 - loss: 1.1801 - val_accuracy: 0.1489 - val_auc: 0.5000 - val_loss: 0.7090
Epoch 7/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/s

Cellule de Code : Évaluation du modèle de Deep Learning

In [29]:

y_pred_dl_proba = model_dl.predict(X_test).ravel()


y_pred_dl = (y_pred_dl_proba > 0.5).astype(int)

print("\n================ RÉSULTATS DU MODÈLE DE DEEP LEARNING ================")
print(f"Score ROC AUC Deep Learning : {roc_auc_score(y_test, y_pred_dl_proba):.4f}")
print("\nRapport de Classification :")
print(classification_report(y_test, y_pred_dl, target_names=['Normal (0)', 'Panne (1)']))

188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

================ RÉSULTATS DU MODÈLE DE DEEP LEARNING ================
Score ROC AUC Deep Learning : 0.5000

Rapport de Classification :
              precision    recall  f1-score   support

  Normal (0)       0.00      0.00      0.00      5144
   Panne (1)       0.14      1.00      0.25       867

    accuracy                           0.14      6011
   macro avg       0.07      0.50      0.13      6011
weighted avg       0.02      0.14      0.04      6011



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Cellule de Code : Sauvegarde du modèle Deep Learning au format .pkl

In [30]:

nom_fichier_dl = "modele_deep_learning_sgd.pkl"

joblib.dump(model_dl, nom_fichier_dl)
print(f"=> Modèle Deep Learning sauvegardé avec succès sous : {nom_fichier_dl}")

=> Modèle Deep Learning sauvegardé avec succès sous : modele_deep_learning_sgd.pkl
